In [1]:
import nba_api.stats.endpoints
import requests
import json
import pandas as pd
import nba_api

In [2]:
# Headers required by stats.nba.com (browser-like)
NBA_STATS_HEADERS = {
    "Accept": "*/*",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "keep-alive",
    "Origin": "https://www.nba.com",
    "Referer": "https://www.nba.com/",
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-site",
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36",
    "sec-ch-ua": '"Not(A:Brand";v="8", "Chromium";v="144", "Brave";v="144"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"macOS"',
}

# Columns shared across measure types (used as merge keys for team combines)
_TEAM_KEY_COLS = [
    "SEASON_YEAR", "TEAM_ID", "TEAM_ABBREVIATION", "TEAM_NAME",
    "GAME_ID", "GAME_DATE", "MATCHUP", "WL",
]
_PLAYER_KEY_COLS = [
    "SEASON_YEAR", "PLAYER_ID", "PLAYER_NAME", "NICKNAME",
    "TEAM_ID", "TEAM_ABBREVIATION", "TEAM_NAME",
    "GAME_ID", "GAME_DATE", "MATCHUP", "WL",
]


def fetch_game_logs(
    entity_type="Team",
    team_id=1610612750,
    player_id=None,
    season="2025-26",
    measure_type="Base",
    period=None,
    game_segment=None,
    season_type="Regular Season",
    date_from="",
    date_to="",
    combine_periods=True,
    timeout=30,
):
    """
    Fetch team or player game logs from stats.nba.com.

    Parameters
    ----------
    entity_type : str
        "Team" or "Player".
    team_id : int
        NBA team ID. Used for team pulls; ignored for player pulls.
    player_id : int or None
        NBA player ID. Required when entity_type="Player".
    season : str
        Season in YYYY-YY format (e.g. "2025-26").
    measure_type : str or list of str
        "Base", "Advanced", "Misc", or a list like ["Base", "Advanced"].
        Lists are only supported for entity_type="Team" — the DataFrames
        are merged on GAME_ID so you get one wide row per game.
    period : int, list of int, or None
        None=full game, 1-4=single quarter, list=multiple quarters.
        combine_periods controls whether multi-quarter stats are summed.
    game_segment : str, list of str, or None
        "First Half", "Second Half", or a list of both.
    season_type : str
        "Regular Season", "Playoffs", etc.
    date_from, date_to : str
        Date filters (e.g. "01/15/2026" or "").
    combine_periods : bool
        When period is a list: True=sum into one row per game, False=keep
        separate rows per quarter.
    timeout : int
        Request timeout in seconds.

    Returns
    -------
    pd.DataFrame
        Game logs. For team pulls with multiple measure_types, columns from
        Advanced/Misc are merged in (duplicates like MIN are kept from Base).
    """
    import time

    # Validate entity_type
    entity_type = entity_type.strip().title()
    if entity_type not in ("Team", "Player"):
        raise ValueError('entity_type must be "Team" or "Player"')

    # Validate measure_type
    valid_measures = ("Base", "Advanced", "Misc")
    if isinstance(measure_type, str):
        measure_type = [measure_type]
    measure_type = [m.strip().title() for m in measure_type]
    for m in measure_type:
        if m not in valid_measures:
            raise ValueError(f'measure_type "{m}" not valid. Use: {valid_measures}')

    # Only team pulls support combining multiple measure types
    if entity_type == "Player" and len(measure_type) > 1:
        raise ValueError("Combining multiple measure types is only supported for Team pulls")

    if entity_type == "Player" and player_id is None:
        raise ValueError("player_id is required when entity_type='Player'")

    # Pick endpoint and ID param
    if entity_type == "Team":
        url = "https://stats.nba.com/stats/teamgamelogs"
        id_param = {"TeamID": str(team_id)}
        key_cols = _TEAM_KEY_COLS
    else:
        url = "https://stats.nba.com/stats/playergamelogs"
        id_param = {"PlayerID": str(player_id)}
        key_cols = _PLAYER_KEY_COLS

    base_params = {
        "DateFrom": date_from,
        "DateTo": date_to,
        "GameSegment": "",
        "ISTRound": "",
        "LastNGames": "0",
        "LeagueID": "00",
        "Location": "",
        "Month": "0",
        "OpponentTeamID": "0",
        "Outcome": "",
        "PORound": "0",
        "PaceAdjust": "N",
        "PerMode": "Totals",
        "PlusMinus": "N",
        "Rank": "N",
        "Season": season,
        "SeasonSegment": "",
        "SeasonType": season_type,
        "ShotClockRange": "",
        "VsConference": "",
        "VsDivision": "",
        **id_param,
    }

    def _one_request(period_val, game_seg, mtype):
        params = base_params.copy()
        params["Period"] = str(period_val)
        params["GameSegment"] = game_seg if game_seg else ""
        params["MeasureType"] = mtype
        r = requests.get(url, headers=NBA_STATS_HEADERS, params=params, timeout=timeout)
        r.raise_for_status()
        data = r.json()
        headers = data["resultSets"][0]["headers"]
        rows = data["resultSets"][0]["rowSet"]
        return pd.DataFrame(rows, columns=headers)

    def _fetch_single_measure(mtype, period, game_segment, combine_periods):
        """Fetch one measure type, handling period/segment logic."""
        # Normalize single-element list
        if isinstance(period, (list, tuple)) and len(period) == 1:
            period = int(period[0])

        # Multiple halves
        if isinstance(game_segment, (list, tuple)) and len(game_segment) >= 1:
            halves = [h.strip() for h in game_segment if str(h).strip() in ("First Half", "Second Half")]
            if not halves:
                raise ValueError('game_segment list must contain only "First Half" and/or "Second Half"')
            dfs = []
            for h in halves:
                df = _one_request(period_val=0, game_seg=h, mtype=mtype)
                df["PERIOD"] = h
                df["PERIOD_TYPE"] = "Half"
                dfs.append(df)
                time.sleep(0.2)
            out = pd.concat(dfs, ignore_index=True)
            return out.sort_values(["GAME_ID", "PERIOD"]).reset_index(drop=True)

        # Multiple quarters
        if isinstance(period, (list, tuple)) and len(period) > 1:
            quarters = [int(q) for q in period if q in (1, 2, 3, 4)]
            if not quarters:
                raise ValueError("period list must contain only 1, 2, 3, 4")
            dfs = []
            for q in quarters:
                df_q = _one_request(period_val=q, game_seg="", mtype=mtype)
                df_q["PERIOD"] = q
                df_q["PERIOD_TYPE"] = "Quarter"
                dfs.append(df_q)
                time.sleep(0.2)
            combined = pd.concat(dfs, ignore_index=True)
            if not combine_periods:
                return combined.sort_values(["GAME_ID", "PERIOD"]).reset_index(drop=True)
            # Sum stats into one row per game
            numeric_cols = combined.select_dtypes(include="number").columns.tolist()
            numeric_cols = [c for c in numeric_cols if c not in ("PERIOD",)]
            id_cols = [c for c in combined.columns if c not in numeric_cols and c not in ("PERIOD", "PERIOD_TYPE")]
            agg_dict = {c: "first" for c in id_cols}
            agg_dict.update({c: "sum" for c in numeric_cols})
            out = combined.groupby("GAME_ID", as_index=False).agg(agg_dict)
            out = out[[c for c in combined.columns if c not in ("PERIOD", "PERIOD_TYPE")]]
            return out

        # Single period or full game
        period_val = 0 if period is None else (int(period) if period in (1, 2, 3, 4) else 0)
        game_seg = (game_segment or "").strip()
        if game_seg and game_seg not in ("First Half", "Second Half"):
            game_seg = "First Half" if "first" in game_seg.lower() else "Second Half"
        out = _one_request(period_val=period_val, game_seg=game_seg, mtype=mtype)
        if period_val in (1, 2, 3, 4):
            out["PERIOD"] = period_val
            out["PERIOD_TYPE"] = "Quarter"
        elif game_seg:
            out["PERIOD"] = game_seg
            out["PERIOD_TYPE"] = "Half"
        else:
            out["PERIOD"] = "Full Game"
            out["PERIOD_TYPE"] = "Full Game"
        return out

    # Single measure type — straightforward
    if len(measure_type) == 1:
        return _fetch_single_measure(measure_type[0], period, game_segment, combine_periods)

    # Multiple measure types (team only) — fetch each, merge on key cols + GAME_ID
    dfs = {}
    for mt in measure_type:
        dfs[mt] = _fetch_single_measure(mt, period, game_segment, combine_periods)
        time.sleep(0.3)

    # Start with the first measure type as the base
    first_mt = measure_type[0]
    result = dfs[first_mt]

    # Determine merge keys (key_cols that exist + PERIOD/PERIOD_TYPE if present)
    merge_keys = [c for c in key_cols if c in result.columns]
    if "PERIOD" in result.columns:
        merge_keys.append("PERIOD")
    if "PERIOD_TYPE" in result.columns:
        merge_keys.append("PERIOD_TYPE")

    # Merge remaining measure types
    for mt in measure_type[1:]:
        df_mt = dfs[mt]
        # Drop columns from df_mt that already exist in result (except merge keys)
        existing_cols = set(result.columns) - set(merge_keys)
        new_cols = [c for c in df_mt.columns if c not in existing_cols or c in merge_keys]
        df_mt = df_mt[new_cols]
        result = result.merge(df_mt, on=merge_keys, how="left")

    return result

In [3]:
# === TEAM examples ===

# Team: traditional box scores, all quarters separate
# df = fetch_game_logs(entity_type="Team", team_id=1610612750, period=[1, 2, 3, 4], combine_periods=False)
df = fetch_game_logs(entity_type="Team", team_id=1610612743, period=[1, 2, 3, 4], combine_periods=False)

# Team: advanced box scores, full game
# df = fetch_game_logs(entity_type="Team", team_id=1610612750, measure_type="Advanced")

# Team: misc box scores, full game
# df = fetch_game_logs(entity_type="Team", team_id=1610612750, measure_type="Misc")

# Team: combined traditional + advanced + misc (merged into one wide DataFrame)
# df = fetch_game_logs(entity_type="Team", team_id=1610612750, measure_type=["Base", "Advanced", "Misc"])

# === PLAYER examples ===

# Player: traditional box scores (Anthony Edwards = 1630162)
# df = fetch_game_logs(entity_type="Player", player_id=1630162, measure_type="Base")

# Player: advanced box scores
# df = fetch_game_logs(entity_type="Player", player_id=1630162, measure_type="Advanced")

# Player: misc box scores
# df = fetch_game_logs(entity_type="Player", player_id=1630162, measure_type="Misc")

print(f"Shape: {df.shape}")
df.head()

ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)

In [ ]:
print(df.columns)

Index(['SEASON_YEAR', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID',
       'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M',
       'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST',
       'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS',
       'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'FGM_RANK',
       'FGA_RANK', 'FG_PCT_RANK', 'FG3M_RANK', 'FG3A_RANK', 'FG3_PCT_RANK',
       'FTM_RANK', 'FTA_RANK', 'FT_PCT_RANK', 'OREB_RANK', 'DREB_RANK',
       'REB_RANK', 'AST_RANK', 'TOV_RANK', 'STL_RANK', 'BLK_RANK', 'BLKA_RANK',
       'PF_RANK', 'PFD_RANK', 'PTS_RANK', 'PLUS_MINUS_RANK', 'AVAILABLE_FLAG'],
      dtype='object')


In [ ]:
df['STOCKS'] = df['STL'] + df['BLK']

# df_1 = df.sort_values(by=['STOCKS', 'GAME_DATE'], ascending=False)
df_1 = df.sort_values(by=['PCT_FG', 'GAME_DATE'], ascending=[True, False])

# df_1[['TEAM_NAME', 'MATCHUP', 'GAME_DATE', 'MATCHUP', 'WL', 'PERIOD', 'STL', 'BLK', 'STOCKS', 'PLUS_MINUS']].reset_index(drop=True)[df_1['STOCKS'] >=8]
df_1[['TEAM_NAME', 'MATCHUP', 'GAME_DATE', 'MATCHUP', 'WL', 'PTS', 'AST', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'PLUS_MINUS']].reset_index(drop=True)

,TEAM_NAME,MATCHUP,GAME_DATE,MATCHUP,WL,PTS,AST,FG3M,FG3A,FG3_PCT,PLUS_MINUS
0,Minnesota Timberwolves,MIN vs. LAC,2026-02-08T00:00:00,MIN vs. LAC,L,59,10,5,28,0.525,-21.0
1,Minnesota Timberwolves,MIN vs. DET,2026-03-28T00:00:00,MIN vs. DET,L,60,15,7,32,0.676,-14.0
2,Minnesota Timberwolves,MIN @ LAC,2026-02-26T00:00:00,MIN @ LAC,W,63,15,5,19,0.850,-5.0
3,Minnesota Timberwolves,MIN vs. GSW,2026-01-25T00:00:00,MIN vs. GSW,L,63,12,9,24,1.110,-22.0
4,Minnesota Timberwolves,MIN @ LAL,2026-03-10T00:00:00,MIN @ LAL,L,68,13,6,32,0.583,-16.0
...,...,...,...,...,...,...,...,...,...,...,...
76,Minnesota Timberwolves,MIN @ MIL,2026-01-13T00:00:00,MIN @ MIL,W,106,29,17,34,1.548,28.0
77,Minnesota Timberwolves,MIN vs. CLE,2026-01-08T00:00:00,MIN vs. CLE,W,106,26,17,31,1.825,17.0
78,Minnesota Timberwolves,MIN @ WAS,2026-01-04T00:00:00,MIN @ WAS,W,108,25,8,28,0.874,29.0
79,Minnesota Timberwolves,MIN vs. UTA,2026-03-18T00:00:00,MIN vs. UTA,W,110,30,13,28,1.429,26.0


In [ ]:
# Most combined free throw attempts (both teams) in a single game — Regular Season
# LeagueGameLog (team rows) has one row per team per game; sum FTA by GAME_ID.
from nba_api.stats.endpoints.leaguegamelog import LeagueGameLog

SEASON = "2025-26"

lg = LeagueGameLog(
    season=SEASON,
    season_type_all_star="Regular Season",
    player_or_team_abbreviation="T",
    timeout=90,
    headers=NBA_STATS_HEADERS,
)
team_logs = lg.get_data_frames()[0]
rs = team_logs[team_logs["GAME_ID"].astype(str).str[2] == "2"].copy()

combined = (
    rs.groupby("GAME_ID", as_index=False)
    .agg(
        GAME_DATE=("GAME_DATE", "first"),
        COMBINED_FTA=("FTA", "sum"),
        MATCHUP=("MATCHUP", "first"),
    )
    .sort_values("COMBINED_FTA", ascending=False)
)

max_fta = int(combined["COMBINED_FTA"].iloc[0])
top = combined[combined["COMBINED_FTA"] == max_fta].reset_index(drop=True)

print(f"{SEASON} Regular Season — highest combined FTA in one game: {max_fta}")
if len(top) > 1:
    print(f"({len(top)} games tied)")
print()

for _, row in top.iterrows():
    gid = row["GAME_ID"]
    per_team = (
        rs[rs["GAME_ID"] == gid][["TEAM_ABBREVIATION", "FTA", "FTM", "MATCHUP"]]
        .sort_values("FTA", ascending=False)
        .reset_index(drop=True)
    )
    print(row["GAME_DATE"], "|", row["MATCHUP"], "| GAME_ID", gid)
    print(per_team.to_string(index=False))
    print()


2025-26 Regular Season — highest combined FTA in one game: 91

2025-10-23 | IND vs. OKC | GAME_ID 0022500005
TEAM_ABBREVIATION  FTA  FTM     MATCHUP
              OKC   51   45   OKC @ IND
              IND   40   30 IND vs. OKC

